# NB02 - Modélisation dimensionelle et ETL

## Objectifs
Construction d'un model dimensionel (schéma en étoile) à partir des données TPC-H en utilisant les tables **silver** et **gold**

1. **Définition de la structure des tables** : Compréhension du fonctionnement de `GENERATED ALWAYS AS IDENTITY` et créer de toutes les tables nécéssaires (silver, gold)
2. **Chargement de la couche silver** : mettre les données des tables **bronzes** vers les tables **silver** avec les transformations et contraintes nécéssaires
3. Créer une dimension SCD Type 2
4. **Chargement de la couche gold** : Remplissage des dimensions et des tables de faits avec de la donnée initiale et incrémentale. (+ utilisation du `MERGE`)

L'objéctif final est d'avoir le schéma **gold** en étoile avec un SCD de Type 2 dimension (garder l'historique complet des changements).

> **Prérequis**: pour la suite du notebook il faut avoir exécuter le setup du Notebook NB01 précédent. 

## Définitions des tables
- **Silver** -> couche d'intégration
- **Gold** -> couche qui implémente le schéma en étoide avec un SCD Type 2 dimension

On utilisera `GENERATED ALWAYS AS IDENTITY` dans Delta, qui permet assigner des valeurs numériques auto-incrémentées.

In [0]:
# récupération du nom du catalog
catalog_name = "demo_" + spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

# Définition du schéma à utiliser
silver_schema = "silver"

# On renseigne les catalogue / schéma à utiliser
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {silver_schema}")

display(f"Catalog : {catalog_name}")
display(f"Schema : {silver_schema}")


In [0]:
spark.sql("DROP TABLE IF EXISTS refined_customer")
spark.sql("DROP TABLE IF EXISTS refined_orders")

## Création des tables **Silver**

On va créer 2 tables :
- `refined_customer` (basée sur TPC-H `customer`)
- `refined_orders` (basée sur TPC-H `orders`)

Pour chacune des tables on va renommer et standardiser les colonnes.

In [0]:
%sql
-- création de refined_customer
CREATE TABLE IF NOT EXISTS refined_customer (
  customer_id INT,        -- Identifiant UNIQUE du client
  name STRING,            -- Nom du client
  address STRING,         -- Adresse du client
  nation_key INT,         -- Code pays 
  phone STRING,           -- Numéro de téléphone
  acct_bal DECIMAL(12,2), -- Solde du compte
  market_segment STRING,  -- Segment de marché
  comment STRING          -- Commentaire
);

In [0]:
%sql
-- création de refined_orders
CREATE TABLE IF NOT EXISTS refined_orders (
  order_id INT,              -- Identifiant UNIQUE de la commande
  customer_id INT,           -- Identifiant du client
  order_status STRING,       -- Statut de la commande
  total_price DECIMAL(12,2), -- Prix total de la commande
  order_date DATE,           -- Date de la commande
  order_priority STRING,     -- Priorité de livraison
  clerk STRING,              -- Nom du personnel
  ship_priority INT,         -- Priorité de livraison
  order_comment STRING       -- Commentaire sur la commande
);

## Création des tables **Gold**

On définie : 
- DimCustomer (avec des attributs SCD Type 2)
- DimDate
- FactOrders

**Etapes** : 
1. `GENERATED ALWAYS AS IDENTITY` pour les clés
2. colonnes supplémentaires dans DimCustomer pour le SCD (ex. start_date, end_date, et is_current)

In [0]:
gold_schema = "gold"

spark.sql(f"USE {gold_schema}")

spark.sql("DROP TABLE IF EXISTS DimCustomer")
spark.sql("DROP TABLE IF EXISTS DimDate")
spark.sql("DROP TABLE IF EXISTS FactOrders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DimCustomer (
  dim_customer_key BIGINT GENERATED ALWAYS AS IDENTITY, -- Identifiant unique pour la dimension
  customer_id INT,        -- Identifiant UNIQUE du client
  name STRING,            -- Nom du client
  address STRING,         -- Adresse du client
  nation_key INT,         -- Code pays 
  phone STRING,           -- Numéro de téléphone
  acct_bal DECIMAL(12,2), -- Solde du compte
  market_segment STRING,  -- Segment de marché
  start_date DATE,        -- Date de début SCD2 début de l'enregistrement
  end_date DATE,          -- Date de fin SCD2 fin de l'enregistrement
  is_current BOOLEAN,     -- Indicateur SCD2 si l'enregistrement est la version en cours
  comment STRING,          -- Commentaire
  CONSTRAINT pk_dim_customer PRIMARY KEY(dim_customer_key)
);

In [0]:
%sql

CREATE TABLE IF NOT EXISTS DimDate
(
  dim_date_key BIGINT GENERATED ALWAYS AS IDENTITY, -- Identifiant unique pour la dimension
  full_date DATE,                                   -- Date complête
  day INT,                                          -- Jour du mois
  month INT,                                        -- Mois de l'année
  year INT,                                         -- Année
  CONSTRAINT pk_dim_date PRIMARY KEY(dim_date_key) RELY
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS FactOrders
(
  fact_orders_key BIGINT GENERATED ALWAYS AS IDENTITY, -- Identifiant unique pour la dimension
  order_id INT,                                        -- Identifiant UNIQUE de la commande
  dim_customer_key BIGINT,                             -- Clé étrangère vers DimCustomer
  dim_date_key BIGINT,                                 -- Clé étrangère vers DimCustomer
  total_price DECIMAL(12,2),                           -- Prix total de la commande
  order_status STRING,                                 -- Statut de la commande
  order_priority STRING,                               -- Priorité de livraison
  clerk STRING,                                        -- Nom du personnel
  ship_priority INT,                                   -- Priorité de livraison
  comment STRING,                                      -- Commentaire
  CONSTRAINT pk_fact_orders PRIMARY KEY(fact_orders_key),
  CONSTRAINT fk_dim_customer FOREIGN KEY(dim_customer_key) REFERENCES DimCustomer(dim_customer_key),
  CONSTRAINT fk_dim_date FOREIGN KEY(dim_date_key) REFERENCES DimDate(dim_date_key)
);

## Chargement des données dans les tables Silver

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {silver_schema}")

In [0]:
%sql
INSERT INTO
  refined_customer
SELECT
  c_custkey as customer_id,
  TRIM(c_name) as name,
  TRIM(c_address) as address,
  c_nationkey as nation_key,
  TRIM(c_phone) as phone,
  CAST(c_acctbal as DECIMAL(12, 2)) as acct_bal,
  TRIM(c_mktsegment) as market_segment,
  TRIM(c_comment) as comment
FROM
  bronze.tpch_customer;

In [0]:
%sql
INSERT INTO
  refined_orders
SELECT
  o_orderkey as order_id,
  o_custkey as customer_id,
  TRIM(o_orderstatus) as order_status,
  CAST(o_totalprice as DECIMAL(12, 2)) as total_price,
  o_orderdate as order_date,
  TRIM(o_orderpriority) as order_priority,
  TRIM(o_clerk) as clerk,
  o_shippriority as ship_priority,
  TRIM(o_comment) as order_comment
FROM
  bronze.tpch_orders;

In [0]:
display(spark.sql("SELECT COUNT(*) AS refined_customer_count from refined_customer"))

display(spark.sql("SELECT COUNT(*) AS refined_orders_count from refined_orders"))

## Chargement initial des tables de Gold

Etapes : 
1. Un chargement initial de `DimCustomer` (tout les clients, customers en tant que current)
2. Création d'entrée pour la table `DimDate` à partir de `refined_orders`
3. alimentation de la table `FactOrders` en lien avec les bonnes dimensions.

In [0]:
spark.sql(f"USE {gold_schema}")

### Alimentation de DimCustomer

On va marquer toutes les lignes avec `start_date = CURRENT_DATE(), end_date = NULL et is_current = TRUE`

In [0]:
spark.sql(f"""
INSERT INTO DimCustomer
(
    customer_id,
    name,
    address,
    nation_key,
    phone,
    acct_bal,
    market_segment,
    comment,
    start_date,
    end_date,
    is_current
)
SELECT
    customer_id,
    name,
    address,
    nation_key,
    phone,
    acct_bal,
    market_segment,
    'comment',
    CURRENT_DATE(),
    NULL,
    TRUE          
FROM {silver_schema}.refined_customer
""")

### Alimentation de DimDate
1. Récupération des valeurs (uniques) de `order_date` de `refined_orders`
2. On va diviser la date en jours, mois, années

In [0]:
spark.sql(f"""
INSERT INTO DimDate
(
    full_date,
    day,
    month,
    year
)
SELECT DISTINCT
    order_date,
    day(order_date),
    month(order_date),
    year(order_date)
FROM {silver_schema}.refined_orders
WHERE order_date IS NOT NULL
""")

### Alimentation de FactOrders
- Relier chaque lignes de refined_orders à `DimCustomer` et `DimDate`
- On fait une jointure sur `customer_id = dc.customer_id AND is_current = TRUE` pour le SCD Type 2, pour s'assurer que on ne récupère que les enrigistrements avec une dimension active.

In [0]:
spark.sql(f"""
INSERT INTO FactOrders
(
  order_id,
  dim_customer_key,
  dim_date_key,
  total_price,
  order_status,
  order_priority,
  clerk,
  ship_priority,
  comment
)
SELECT
    ro.order_id,
    dc.dim_customer_key,
    dd.dim_date_key,
    ro.total_price,
    ro.order_status,
    ro.order_priority,
    ro.clerk,
    ro.ship_priority,
    ro.order_comment
FROM {silver_schema}.refined_orders ro
JOIN DimCustomer dc 
  ON ro.customer_id = dc.customer_id
  AND dc.is_current = TRUE
JOIN DimDate dd 
  ON ro.order_date = dd.full_date
""")

### validation

In [0]:
display(spark.sql("SELECT 'DimCustomer' AS table_name, COUNT(*) AS recourd_count FROM DimCustomer"))
display(spark.sql("SELECT 'DimDate' AS table_name, COUNT(*) AS recourd_count FROM DimDate"))
display(spark.sql("SELECT 'FactOrders' AS table_name, COUNT(*) AS recourd_count FROM FactOrders"))

## MAJ Incrémentales (SCD Type 2 MERGE)

1. **Fermer** un ancien enrigistrement dans DimCustomer (end_date = current_date, is_current = false)
2. **Insert** un nouvel enregistrement (start_date = current_date, end_date = null et is_current = true)

### Création d'une simulation d'un changement
- Un client (ID=101) a un changement d'adresse
- un nouveau client (ID=99999999)

In [0]:
spark.sql("""
create or replace temp view incremental_customer_updates as
SELECT 101 AS customer_id,
       'customer_101' AS name,
       'address_101' AS address,
       77 AS nation_key,
       '+33199999999' AS phone,
       CAST(999.99 AS decimal(12, 2)) AS acct_bal,
       'NEW SEGMENT' AS market_segment,
       'comment_101' AS comment
UNION ALL
SELECT 999999,
'Nouveau',
'123 Rue Nouvelle',
90,
'+33199999999',
CAST(500.99 AS decimal(12, 2)),
'Nouveau Marché',
'Nouveau client'
""")

display(spark.sql("SELECT * FROM incremental_customer_updates"))

### Opération de MERGE pour SCD Type 2

In [0]:
merge_sql = f"""
WITH staged_changes AS (
        SELECT
            i.customer_id,
            i.name,
            i.address,
            i.nation_key,
            i.phone,
            i.acct_bal,
            i.market_segment,
            i.comment
        FROM incremental_customer_updates i
)
MERGE INTO DimCustomer AS t
USING staged_changes AS s
ON t.customer_id = s.customer_id AND t.is_current = TRUE
WHEN MATCHED THEN
    UPDATE SET
        t.is_current = FALSE,
        t.end_date = CURRENT_DATE()
WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        name,
        address,
        nation_key,
        phone,
        acct_bal,
        market_segment,
        comment,
        start_date,
        end_date,
        is_current
    )
    VALUES (
        s.customer_id,
        s.name,
        s.address,
        s.nation_key,
        s.phone,
        s.acct_bal,
        s.market_segment,
        s.comment,
        CURRENT_DATE(),
        NULL,
        TRUE
    );
"""

spark.sql(merge_sql)


In [0]:
display(spark.sql("SELECT * FROM DimCustomer where customer_id = 101"))

display(spark.sql("SELECT * FROM DimCustomer where customer_id = 99999"))

### Top 10 des dépenses par marché

In [0]:
display(spark.sql("""
SELECT
    dc.market_segment,
    SUM(f.total_price) as total_spent
FROM FactOrders f
JOIN DimCustomer dc 
  ON f.dim_customer_key = dc.dim_customer_key
GROUP BY
    dc.market_segment
ORDER BY
    total_spent DESC
LIMIT 10               
"""))

## Netoyage
- suppréssion des tables silver
- suppréssion des tables gold

(désactiver par défaut)

In [0]:
%sql
DROP TABLE IF EXISTS silver.refined_customer;
DROP TABLE IF EXISTS silver.refined_orders;

DROP TABLE IF EXISTS gold.DimCustomer;
DROP TABLE IF EXISTS gold.DimDate;
DROP TABLE IF EXISTS gold.FactOrders;